In [12]:
import os
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# ==========================
# CONFIG
# ==========================

BASE_URLS = {f"Season_{i}": f"https://results.hyrox.com/season-{i}/" for i in range(1, 9)}
SAVE_ROOT = r"Datasets\Hyrox"

# ==========================
# DRIVER
# ==========================

def create_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=options)

# ==========================
# VERIFICATION
# ==========================

def verify_all_downloads():

    print("\n==============================")
    print("VERIFYING HYROX DATASET")
    print("==============================")

    for season, base_url in BASE_URLS.items():

        print(f"\n========== CHECKING {season} ==========")

        season_folder = os.path.join(SAVE_ROOT, season)

        if not os.path.exists(season_folder):
            print(f"❌ Season folder missing: {season}")
            continue

        local_files = set(os.listdir(season_folder))

        driver = create_driver()
        driver.get(base_url)

        try:
            WebDriverWait(driver, 20).until(
                
                EC.presence_of_element_located((By.ID, "default-lists-event_main_group"))
            )
        except TimeoutException:
            print(f"❌ Could not load season page: {season}")
            driver.quit()
            continue

        race_dropdown = Select(driver.find_element(By.ID, "default-lists-event_main_group"))
        races = [(opt.text.strip(), opt.get_attribute("value"))
                 for opt in race_dropdown.options]

        missing_races = []
        missing_divisions = []

        for race_name, race_value in races:

            safe_race = race_name.replace(" ", "_").replace("/", "-")

            print(f"Checking race: {race_name}")

            driver.get(base_url)
            time.sleep(2)

            Select(driver.find_element(By.ID, "default-lists-event_main_group"))\
                .select_by_value(race_value)
            time.sleep(2)

            division_dropdown = Select(driver.find_element(By.ID, "default-lists-event"))
            divisions = [opt.text.strip() for opt in division_dropdown.options]

            race_files = [f for f in local_files if f.startswith(safe_race + "_")]

            if not race_files:
                missing_races.append(race_name)

            race_missing_divs = []

            for div in divisions:
                safe_div = div.replace(" ", "_").replace("/", "-")
                expected_filename = f"{safe_race}_{safe_div}.csv"

                if expected_filename not in local_files:
                    race_missing_divs.append(div)

            if race_missing_divs:
                missing_divisions.append((race_name, race_missing_divs))

        driver.quit()

        # REPORT
        if not missing_races:
            print("✅ No missing races.")
        else:
            print("\n❌ Missing races:")
            for r in missing_races:
                print(f"   - {r}")

        if not missing_divisions:
            print("✅ All divisions present.")
        else:
            print("\n⚠ Missing divisions:")
            for race, divs in missing_divisions:
                print(f"\n  {race}")
                for d in divs:
                    print(f"     - {d}")

        print(f"\n========== DONE {season} ==========")

    print("\n==============================")
    print("VERIFICATION COMPLETE")
    print("==============================")

# ==========================
# RUN
# ==========================

verify_all_downloads()


VERIFYING HYROX DATASET

========== CHECKING Season_1 ==========
Checking race: World Championships
Checking race: 2019 Oberhausen
Checking race: 2019 Karlsruhe
Checking race: 2019 Nürnberg
Checking race: 2019 Hannover
Checking race: 2018 Stuttgart
Checking race: 2018 Wien
Checking race: 2018 Essen
Checking race: 2018 Hamburg
Checking race: 2018 Leipzig
✅ No missing races.
✅ All divisions present.

========== DONE Season_1 ==========

========== CHECKING Season_2 ==========
Checking race: 2020 Elite 12
Checking race: 2020 Karlsruhe
Checking race: 2020 Dallas
Checking race: 2020 Hannover
Checking race: 2020 Chicago
Checking race: 2019 New York
Checking race: 2019 Frankfurt
Checking race: 2019 Wien
Checking race: 2019 Hamburg
Checking race: 2019 Essen
Checking race: 2019 Leipzig
Checking race: 2019 Miami
✅ No missing races.
✅ All divisions present.

========== DONE Season_2 ==========

========== CHECKING Season_3 ==========
Checking race: 2021 Dallas
Checking race: 2021 Chicago
Checkin

StaleElementReferenceException: Message: stale element reference: stale element not found in the current frame
  (Session info: chrome=145.0.7632.117); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7a42faa55
	0x7ff7a4058630
	0x7ff7a3ded75d
	0x7ff7a3df56c1
	0x7ff7a3df8b7e
	0x7ff7a3e96ce1
	0x7ff7a3e719da
	0x7ff7a3e9591c
	0x7ff7a3e39098
	0x7ff7a3e39f83
	0x7ff7a4327810
	0x7ff7a4321afd
	0x7ff7a4342c1a
	0x7ff7a4073345
	0x7ff7a407b81c
	0x7ff7a4061924
	0x7ff7a4061ad6
	0x7ff7a4047e47
	0x7ffa940ae8d7
	0x7ffa94c6c40c


In [18]:
import os
import re
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException

# ==========================
# CONFIG
# ==========================
BASE_URLS = {f"Season_{i}": f"https://results.hyrox.com/season-{i}/" for i in range(1, 9)}
SAVE_ROOT = r"Datasets\Hyrox"

# ==========================
# DRIVER
# ==========================
def create_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=options)

# ==========================
# HELPERS
# ==========================
def safe_name(name: str) -> str:
    """Make a string safe for filenames."""
    return re.sub(r"[^\w\-]", "_", name.strip())

def get_stable_select(driver, element_id, timeout=10, retries=5):
    """Refetch the <select> element until it is stable."""
    for _ in range(retries):
        try:
            elem = WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located((By.ID, element_id))
            )
            if elem.tag_name.lower() == "select":
                return Select(elem)
        except StaleElementReferenceException:
            time.sleep(0.5)
    raise Exception(f"Could not get stable <select> for {element_id}")

# ==========================
# EXTRA CSV CHECK
# ==========================
def check_extra_csvs():
    print("\n==============================")
    print("CHECKING FOR EXTRA CSV FILES")
    print("==============================\n")

    driver = create_driver()

    for season, base_url in BASE_URLS.items():
        print(f"\n========== CHECKING {season} ==========")

        season_folder = os.path.join(SAVE_ROOT, season)
        if not os.path.exists(season_folder):
            print(f"❌ Season folder missing: {season}")
            continue

        local_files = set(os.listdir(season_folder))
        used_files = set()

        driver.get(base_url)
        time.sleep(2)

        # -----------------------------
        # Get list of races
        # -----------------------------
        try:
            race_select = get_stable_select(driver, "default-lists-event_main_group")
            races = [(opt.text.strip(), opt.get_attribute("value")) 
                     for opt in race_select.options if "team relay" not in opt.text.lower()]
        except TimeoutException:
            print(f"❌ Could not load races for {season}")
            continue

        for race_name, race_value in races:
            safe_race = safe_name(race_name)

            # Select the race with retry
            for attempt in range(3):
                try:
                    race_select = get_stable_select(driver, "default-lists-event_main_group")
                    race_select.select_by_value(race_value)
                    time.sleep(1)
                    break
                except StaleElementReferenceException:
                    time.sleep(1)

            # Get divisions for this race
            try:
                division_select = get_stable_select(driver, "default-lists-event")
                divisions = [opt.text.strip() for opt in division_select.options
                             if "team relay" not in opt.text.lower()]
            except Exception:
                divisions = []

            for div in divisions:
                safe_div = safe_name(div)
                used_files.add(f"{safe_race}_{safe_div}.csv")

        # -----------------------------
        # Check for extra CSVs
        # -----------------------------
        extra_files = set()
        for f in local_files:
            if f.lower().endswith(".csv") and f not in used_files:
                if "team_relay" not in f.lower():  # skip team relay
                    extra_files.add(f)

        if not extra_files:
            print("✅ No extra CSVs found.")
        else:
            print("⚠ Extra CSVs detected:")
            for f in sorted(extra_files):
                print(f"   - {f}")

        print(f"\n========== DONE {season} ==========")

    driver.quit()
    print("\n==============================")
    print("EXTRA CSV CHECK COMPLETE")
    print("==============================")

# ==========================
# RUN
# ==========================
check_extra_csvs()


CHECKING FOR EXTRA CSV FILES


========== CHECKING Season_1 ==========
✅ No extra CSVs found.

========== DONE Season_1 ==========

========== CHECKING Season_2 ==========
✅ No extra CSVs found.

========== DONE Season_2 ==========

========== CHECKING Season_3 ==========
✅ No extra CSVs found.

========== DONE Season_3 ==========

========== CHECKING Season_4 ==========
⚠ Extra CSVs detected:
   - U.S._Championships_of_Fitness_2021_-_Chicago_HYROX_DOUBLES.csv
   - U.S._Championships_of_Fitness_2021_-_Chicago_HYROX_ELITE.csv
   - U.S._Championships_of_Fitness_2021_-_Chicago_HYROX_PRO.csv

========== DONE Season_4 ==========

========== CHECKING Season_5 ==========
✅ No extra CSVs found.

========== DONE Season_5 ==========

========== CHECKING Season_6 ==========
✅ No extra CSVs found.

========== DONE Season_6 ==========

========== CHECKING Season_7 ==========
⚠ Extra CSVs detected:
   - 2025_St._Gallen_HYROX.csv
   - 2025_St._Gallen_HYROX_DOUBLES.csv
   - 2025_St._Gallen_HYROX_PRO.